# Airbnb Price Prediction Machine Learning Project

## Problem Statement
Airbnb is an online marketplace for lodging and tourism. Price is a key factor for customers when booking. The goal is to build models to predict the price of a new Airbnb property based on existing property data.

## Objective
* Build both Simple Linear Regression and Multiple Linear Regression models
* Predict the price of Airbnb listings

### Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

### Step 2: Load Data

In [ ]:
df = pd.read_csv('../data/AirBNB.csv', low_memory=False)
print("First 5 rows:")
display(df.head())
print(f"Dataset Shape: {df.shape}")
print("Columns:", df.columns.tolist())

### Step 3: Understand Data

In [ ]:
df.info()
display(df.describe())

### Step 4: Data Cleaning

In [ ]:
print("Missing values:")
print(df.isnull().sum())

df.drop_duplicates(inplace=True)

# Handle missing values for target
df = df.dropna(subset=['log_price'])

# Convert log_price to numeric just in case
df['log_price'] = pd.to_numeric(df['log_price'], errors='coerce')

# Drop irrelevant columns
cols_to_drop = ['id', 'host_since']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

# CRITICAL: Fix mixed type error (bool vs str) in categorical features
cat_cols = ['room_type', 'cancellation_policy', 'city', 'host_has_profile_pic', 'host_identity_verified', 'instant_bookable', 'cleaning_fee']
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).replace('nan', 'missing')

### Step 5: Basic EDA

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['log_price'], bins=30, kde=True)
plt.title('Log Price Distribution')
plt.show()

if 'room_type' in df.columns:
    plt.figure(figsize=(10, 5))
    sns.countplot(data=df, x='room_type')
    plt.title('Room Type Distribution')
    plt.show()

### Step 6: Outlier Detection

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(x=df['log_price'], color='orange')
plt.title('Outliers in Log Price')
plt.show()

### Step 7: Bivariate Analysis

In [ ]:
if 'room_type' in df.columns:
    plt.figure(figsize=(10, 5))
    sns.barplot(x='room_type', y='log_price', data=df)
    plt.title('Average Log Price by Room Type')
    plt.show()

### Step 8: Feature Engineering

In [ ]:
# Encoding will be handled by the Pipeline during modeling.

### Step 9: Feature Selection

In [ ]:
num_cols = ['accommodates', 'bathrooms', 'bedrooms', 'beds', 'number_of_reviews', 'review_scores_rating']
cat_cols = ['room_type', 'cancellation_policy', 'city', 'host_has_profile_pic', 'host_identity_verified', 'instant_bookable', 'cleaning_fee']

numerical_features = [c for c in num_cols if c in df.columns]
categorical_features = [c for c in cat_cols if c in df.columns]

X = df[numerical_features + categorical_features]
y = df['log_price']

plt.figure(figsize=(10, 8))
sns.heatmap(df[numerical_features + ['log_price']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

### Step 10: Simple Linear Regression

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

simple_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('regressor', LinearRegression())
])

simple_pipe.fit(X_train[['accommodates']], y_train)
y_pred_s = simple_pipe.predict(X_test[['accommodates']])

print(f"Simple LR R2: {r2_score(y_test, y_pred_s):.4f}")
print(f"Simple LR MAE: {mean_absolute_error(y_test, y_pred_s):.4f}")

### Step 11: Multiple Linear Regression

In [ ]:
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, numerical_features),
    ('cat', cat_transformer, categorical_features)
])

multiple_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

multiple_pipe.fit(X_train, y_train)
y_pred_m = multiple_pipe.predict(X_test)

print(f"Multiple LR R2: {r2_score(y_test, y_pred_m):.4f}")
print(f"Multiple LR MAE: {mean_absolute_error(y_test, y_pred_m):.4f}")

### Step 12: Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Simple LR', 'Multiple LR'],
    'R2 Score': [r2_score(y_test, y_pred_s), r2_score(y_test, y_pred_m)]
})
display(results)
print("Multiple Linear Regression performs better due to inclusion of more informative features.")

### Step 13: Prediction

In [ ]:
sample = X_test.iloc[[0]]
pred = multiple_pipe.predict(sample)[0]
actual = y_test.iloc[0]
print(f"Sample Input: {sample.to_dict('records')[0]}")
print(f"Actual Log Price: {actual:.4f}")
print(f"Predicted Log Price: {pred:.4f}")
print(f"Approximate price in USD: ${np.exp(pred):.2f}")

### Step 14: Visualization

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_m, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted Log Price')
plt.show()

### Step 15: Final Output

The model effectively predicts the log_price of Airbnb listings. The Multiple Linear Regression model is superior and can be used for price estimation of new listings.